**Mount Drive (Google Colab)**

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Dataset Loading and Inspection**

In [12]:
import pandas as pd

heart_data = pd.read_csv('drive/MyDrive/Datasets For ML/Heart Failure Prediction.csv')

print("Shape:", heart_data.shape)
print("\nDuplicated:", heart_data.duplicated().sum())
print("\nMissing Values:", heart_data.isna().sum().sum())
print("\nFirst 2 Rows:\n")
heart_data.head(2)

Shape: (299, 13)

Duplicated: 0

Missing Values: 0

First 2 Rows:



,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1


**Feature–Label Separation**

**Assuming:**

Target column = DEATH_EVENT

0 = No Heart Disease

1 = Heart Disease

In [13]:
X = heart_data.drop('DEATH_EVENT', axis=1)
Y = heart_data['DEATH_EVENT']

**Train–Test Split and Feature Scaling**

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("First 2 Rows of Scaled X_train:\n")
display(pd.DataFrame(X_train[:2], columns=X.columns))

First 2 Rows of Scaled X_train:



,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time
0,-0.269050,1.110696,-0.200735,-0.900337,0.176528,-0.770281,-1.004722,-0.360437,0.559915,-1.333818,-0.682831,-0.467847
1,-0.706883,-0.900337,-0.534318,1.110696,1.847425,-0.770281,1.051685,-0.544467,-0.345802,0.749728,-0.682831,-1.359167


**Model Comparison: SVC with Different Kernels**

**Test:** Linear, RBF, Polynomial

In [15]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

kernels = ['linear', 'rbf', 'poly']
C_values = [0.1, 1, 10]

results = []

for kernel in kernels:
    for C in C_values:
        model = SVC(
            kernel=kernel,
            C=C,
            gamma='scale',
            probability=True
        )

        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_test)
        Y_proba = model.predict_proba(X_test)[:,1]

        results.append([
            kernel,
            C,
            accuracy_score(Y_test, Y_pred),
            f1_score(Y_test, Y_pred),
            roc_auc_score(Y_test, Y_proba)
        ])

results_df = pd.DataFrame(
    results,
    columns=['Kernel','C','Accuracy','F1','ROC_AUC']
)

print(results_df)

   Kernel     C  Accuracy        F1   ROC_AUC
0  linear   0.1  0.816667  0.645161  0.854942
1  linear   1.0  0.783333  0.551724  0.856226
2  linear  10.0  0.800000  0.625000  0.851091
3     rbf   0.1  0.683333  0.000000  0.852375
4     rbf   1.0  0.766667  0.533333  0.844673
5     rbf  10.0  0.733333  0.529412  0.768935
6    poly   0.1  0.683333  0.000000  0.871630
7    poly   1.0  0.766667  0.461538  0.866496
8    poly  10.0  0.766667  0.533333  0.780488


**NuSVC Model**

In [16]:
from sklearn.svm import NuSVC

nu_model = NuSVC(kernel='rbf', nu=0.5, probability=True)
nu_model.fit(X_train, Y_train)

Y_pred_nu = nu_model.predict(X_test)
Y_proba_nu = nu_model.predict_proba(X_test)[:,1]

print("NuSVC Accuracy:", accuracy_score(Y_test, Y_pred_nu))
print("NuSVC F1 Score:", f1_score(Y_test, Y_pred_nu))
print("NuSVC ROC-AUC:", roc_auc_score(Y_test, Y_proba_nu))

NuSVC Accuracy: 0.7666666666666667
NuSVC F1 Score: 0.5333333333333333
NuSVC ROC-AUC: 0.8446726572528882


**LinearSVC Model**

In [17]:
from sklearn.svm import LinearSVC

linear_model = LinearSVC(C=1, max_iter=5000)
linear_model.fit(X_train, Y_train)

Y_pred_linear = linear_model.predict(X_test)
Y_proba_linear = linear_model.decision_function(X_test)

print("LinearSVC Accuracy:", accuracy_score(Y_test, Y_pred_linear))
print("LinearSVC F1 Score:", f1_score(Y_test, Y_pred_linear))
print("LinearSVC ROC-AUC:", roc_auc_score(Y_test, Y_proba_linear))

LinearSVC Accuracy: 0.8166666666666667
LinearSVC F1 Score: 0.6666666666666666
LinearSVC ROC-AUC: 0.8639281129653402


**Identify Best Model from Results**

In [18]:
max_svc_roc_auc = results_df['ROC_AUC'].max()
best_svc_params = results_df.loc[results_df['ROC_AUC'].idxmax()]
nu_svc_roc_auc = roc_auc_score(Y_test, Y_proba_nu)
linear_svc_roc_auc = roc_auc_score(Y_test, Y_proba_linear)

overall_max_roc_auc = max(max_svc_roc_auc, nu_svc_roc_auc, linear_svc_roc_auc)

print("Maximum ROC-AUC among all models:", overall_max_roc_auc)

if overall_max_roc_auc == max_svc_roc_auc:
    print("Best Model: SVC")
    print(f"Kernel: {best_svc_params['Kernel']}, C: {best_svc_params['C']}")
elif overall_max_roc_auc == nu_svc_roc_auc:
    print("Best Model: NuSVC")
    print("Parameters: kernel='rbf', nu=0.5")
elif overall_max_roc_auc == linear_svc_roc_auc:
    print("Best Model: LinearSVC")
    print("Parameters: C=1")

Maximum ROC-AUC among all models: 0.871630295250321
Best Model: SVC
Kernel: poly, C: 0.1


**Train Final Model with Best Parameters**

In [19]:
final_model = SVC(kernel='poly', C=0.1, gamma='scale', probability=True)
final_model.fit(X_train, Y_train)

SVC(C=0.1, kernel='poly', probability=True)

**Model Evaluation**

In [20]:
from sklearn.metrics import confusion_matrix, classification_report

Y_pred_final = final_model.predict(X_test)
Y_proba_final = final_model.predict_proba(X_test)[:,1]

print("Final Accuracy:", accuracy_score(Y_test, Y_pred_final))
print("Final F1 Score:", f1_score(Y_test, Y_pred_final, zero_division=0))
print("Final ROC-AUC:", roc_auc_score(Y_test, Y_proba_final))
print("\nConfusion Matrix:\n", confusion_matrix(Y_test, Y_pred_final))
print("\nClassification Report:\n", classification_report(Y_test, Y_pred_final, zero_division=0))


Final Accuracy: 0.6833333333333333
Final F1 Score: 0.0
Final ROC-AUC: 0.871630295250321

Confusion Matrix:
 [[41  0]
 [19  0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.68      1.00      0.81        41
           1       0.00      0.00      0.00        19

    accuracy                           0.68        60
   macro avg       0.34      0.50      0.41        60
weighted avg       0.47      0.68      0.55        60



**Conclusion**

Different SVM models (SVC, NuSVC, LinearSVC) were compared and the best model was selected based on ROC-AUC score. The RBF kernel showed better performance for the dataset. The final model achieved good classification results and PCA visualization showed clear class separation.
